# Boardroom: Fine-Tuning Llama 3.1 8B with QLoRA via Unsloth

**Goal:** Fine-tune a single Llama 3.1 8B model to role-play 6 fictional characters, conditioned via system prompt.

**What you'll learn:**
1. How QLoRA works (4-bit quantization + Low-Rank Adaptation)
2. How to structure chat-format training data for Llama 3.1
3. How to configure LoRA hyperparameters and understand what each one does
4. How to train with HuggingFace's SFTTrainer
5. How to evaluate and export your model to GGUF for local deployment via Ollama

**Requirements:** Google Colab with T4 GPU (free tier works)

## 1. Setup & Installation

Unsloth is a library that makes fine-tuning LLMs 2-5x faster with 70% less memory. It achieves this through custom CUDA kernels and intelligent memory management. Installing it pulls in all the dependencies we need: `transformers`, `peft` (for LoRA), `trl` (for SFTTrainer), and `bitsandbytes` (for quantization).

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## 2. Load the Base Model

We're using `unsloth/llama-3.1-8b-instruct-bnb-4bit` — a pre-quantized version of Meta's Llama 3.1 8B Instruct model.

**What's happening here:**
- **4-bit quantization (NF4):** The model's weights are stored in 4 bits instead of 16 bits. This is possible because neural network weights follow a roughly normal distribution, and NF4 (Normal Float 4) is optimized for this. This cuts memory from ~16GB to ~5GB — making it fit on a free Colab T4 (16GB VRAM).
- **Instruct variant:** We use the instruction-tuned version (not the base model) because it already knows how to follow system prompts and have conversations. We're adding character personality on top of that capability.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
MAX_SEQ_LENGTH = 1024  # Our examples avg ~350 tokens, max ~2000. 1024 covers nearly all.
DTYPE = None           # Auto-detect: float16 for T4, bfloat16 for A100
LOAD_IN_4BIT = True    # Use 4-bit quantization to fit in Colab

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3.1-8b-instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded. Parameters: {model.num_parameters():,}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. Configure LoRA Adapters

**LoRA (Low-Rank Adaptation)** is the key technique that makes fine-tuning feasible. Instead of updating all 8 billion parameters, we freeze the original weights and inject small trainable matrices into specific layers.

**How it works:**
- A weight matrix W (e.g., 4096×4096 = 16M params) is decomposed into two small matrices: A (4096×r) and B (r×4096)
- With r=16, that's only 4096×16 + 16×4096 = 131K params instead of 16M — a **99.2% reduction**
- During training, only A and B are updated. The original W stays frozen.
- At inference: output = W·x + B·A·x (the adapter adds a correction to the frozen weights)

**Hyperparameters explained:**
- `r=16` — **Rank**: How many dimensions the adapter uses. Higher = more capacity but more memory. 16 is a good starting point for character voice (we're learning style, not new knowledge).
- `lora_alpha=32` — **Scaling factor**: Controls how much the adapter affects the output. Rule of thumb: alpha = 2×r. Higher alpha = adapter has more influence.
- `lora_dropout=0.05` — **Regularization**: Randomly zeros adapter outputs during training to prevent overfitting. Important with our small dataset (~800 examples).
- `target_modules` — **Which layers to adapt**: We target all the attention layers (q/k/v/o projections) and the MLP layers (gate/up/down projections). This gives the adapter enough reach to learn character-specific response patterns.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",       # MLP layers
    ],
    bias="none",        # Don't train bias terms (standard practice)
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing (30% less VRAM)
    random_state=42,
)

# See how few parameters we're actually training
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 4. Load & Inspect the Training Dataset

Upload your `boardroom_train.jsonl` file (generated by `build_dataset.py`) to Colab. Each line is a JSON object with a `messages` array in Llama 3.1 chat format:

```json
{
  "messages": [
    {"role": "system", "content": "You are Harvey Specter..."},
    {"role": "user", "content": "What do you think of this idea: ..."},
    {"role": "assistant", "content": "actual dialogue from transcript"}
  ]
}
```

The key insight: **the system prompt is what conditions the model to switch characters.** Every training example for Harvey has the same Harvey system prompt, every example for Logan has Logan's prompt, etc. The model learns to associate each system prompt with a distinct voice.

In [ ]:
from google.colab import files
from datasets import load_dataset

# Upload your training data
print("Upload boardroom_train.jsonl:")
uploaded = files.upload()

# Load the dataset
dataset = load_dataset("json", data_files="boardroom_train.jsonl", split="train")
print(f"\nDataset size: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")

# Peek at one example
print("\n--- Sample entry ---")
import json
sample = dataset[0]
print(json.dumps(sample, indent=2)[:500])

In [ ]:
# Inspect the dataset: per-character distribution and token lengths
from collections import Counter

def get_character_name(messages):
    """Extract character name from system prompt."""
    system = messages[0]["content"]
    if "You are " in system:
        after = system.split("You are ")[1]
        for delim in [" from ", ". "]:
            if delim in after:
                return after.split(delim)[0]
    return "Unknown"

# Character distribution
characters = [get_character_name(ex["messages"]) for ex in dataset]
char_counts = Counter(characters)
print("Per-character example counts:")
for char, count in sorted(char_counts.items()):
    bar = "█" * (count // 2)
    print(f"  {char:20s} {count:4d} {bar}")

# Token length distribution
print("\nToken lengths (assistant responses):")
lengths = []
for ex in dataset:
    assistant_text = ex["messages"][-1]["content"]
    tokens = tokenizer.encode(assistant_text)
    lengths.append(len(tokens))

import statistics
print(f"  Mean: {statistics.mean(lengths):.0f} tokens")
print(f"  Median: {statistics.median(lengths):.0f} tokens")
print(f"  Min: {min(lengths)}, Max: {max(lengths)}")

# Check full sequence lengths (system + user + assistant)
full_lengths = []
for ex in dataset:
    full_text = tokenizer.apply_chat_template(ex["messages"], tokenize=True)
    full_lengths.append(len(full_text))

over_limit = sum(1 for l in full_lengths if l > MAX_SEQ_LENGTH)
print(f"\nFull sequence lengths:")
print(f"  Mean: {statistics.mean(full_lengths):.0f}, Max: {max(full_lengths)}")
print(f"  Over {MAX_SEQ_LENGTH} limit: {over_limit} examples")

## 5. Tokenize with Chat Template

Llama 3.1 uses a specific chat template format with special tokens. We need to convert our `messages` list into the tokenized format the model expects:

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are Harvey Specter...<|eot_id|><|start_header_id|>user<|end_header_id|>

What do you think...<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I don't play the odds...<|eot_id|>
```

The `apply_chat_template` method from the tokenizer handles this automatically. We just need to wrap it in a formatting function for the trainer.

In [ ]:
def formatting_func(examples):
    """Convert messages list to formatted chat strings for the trainer."""
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,  # Don't add trailing assistant prompt
        )
        texts.append(text)
    return {"text": texts}

# Apply formatting to dataset
dataset = dataset.map(formatting_func, batched=True)

# Verify: show what the model will actually see
print("--- Formatted training example ---")
print(dataset[0]["text"][:600])
print("...")

## 6. Training with SFTTrainer

**SFT (Supervised Fine-Tuning)** is the standard approach for teaching a model to produce specific outputs. The trainer takes our formatted examples and optimizes the LoRA adapter weights to minimize the difference between the model's output and the actual character dialogue.

**Hyperparameters explained:**
- `per_device_train_batch_size=2` — How many examples to process at once. Limited by GPU memory. T4 can handle 2 with our setup.
- `gradient_accumulation_steps=4` — Accumulate gradients over 4 mini-batches before updating weights. **Effective batch size = 2 × 4 = 8.** Larger effective batch size = more stable training.
- `num_train_epochs=3` — How many times to see the full dataset. With ~800 examples, 3 epochs ≈ ~300 optimizer steps. Too few = underfitting, too many = overfitting (the model memorizes responses instead of learning style).
- `learning_rate=2e-4` — How aggressively to update weights. 2e-4 is standard for QLoRA. Too high = unstable, too low = slow/no learning.
- `warmup_steps=10` — Gradually increase learning rate from 0 for the first 10 steps. Prevents early instability.
- `weight_decay=0.01` — L2 regularization to prevent overfitting. Penalizes large weight values.
- `logging_steps=5` — Print loss every 5 steps so we can monitor training progress.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  # Don't pack multiple examples into one sequence (cleaner for chat)
    args=TrainingArguments(
        # Batch size — reduced to 1 for T4 memory, compensated with higher accumulation
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,  # effective batch size still = 8

        # Training duration
        num_train_epochs=3,

        # Learning rate
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="cosine",  # Gradually decay LR (better than constant)

        # Regularization
        weight_decay=0.01,

        # Precision
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        # Logging & saving
        logging_steps=5,
        save_steps=50,
        save_total_limit=2,
        output_dir="boardroom-checkpoints",

        # Misc
        optim="adamw_8bit",  # 8-bit Adam saves memory
        seed=42,
    ),
)

print("Trainer configured. Ready to train.")

In [ ]:
# Check memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Memory reserved: {start_gpu_memory} GB / {max_memory} GB")
print(f"\nStarting training...")

# Train!
trainer_stats = trainer.train()

# Report
print(f"\n--- Training complete ---")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")

## 7. Evaluate: Base Model vs Fine-Tuned

The real test — does each character actually sound different? We'll send the same prompt to each character and compare the fine-tuned model's responses against the base model's responses.

**What to look for:**
- Does Harvey use legal/deal language? Does he sound dismissive?
- Does Logan use short, brutal sentences?
- Does Noddy say something confidently nonsensical?
- Are the characters distinguishable from each other, or do they all sound the same?

In [ ]:
TEST_PITCH = "We should build an app that delivers tacos via drone to people stuck in traffic."

CHARACTER_PROMPTS = {
    "Harvey Specter": (
        "You are Harvey Specter from the TV show Suits. You are a senior partner at a top law firm. "
        "You see everything through a legal and deal-making lens. You are obsessed with winning and "
        "deeply dismissive of weakness. You speak in sharp, confident one-liners. You reference "
        "deals, leverage, and closing. You never show vulnerability. Keep responses to 2-3 sentences."
    ),
    "Bob the Builder": (
        "You are Bob the Builder. You see everything through a logistics and construction lens. "
        "You are enthusiastic, optimistic, and accidentally useful. You use construction metaphors "
        "constantly. You ask 'Can we build it?' rhetorically. You break problems down into "
        "practical building steps. Keep responses to 2-3 sentences."
    ),
    "Logan Roy": (
        "You are Logan Roy from the TV show Succession. You see everything through a lens of power, "
        "control, and legacy. You are brutal and terse. You use short, cutting sentences. You dismiss "
        "everyone around you as incompetent. You reference media empires and power dynamics. "
        "You call people 'kid' condescendingly. Keep responses to 2-3 sentences."
    ),
    "Ross Geller": (
        "You are Ross Geller from the TV show Friends. You are over-analytical and go on academic "
        "tangents. You get defensive easily. You bring up paleontology and dinosaurs in unrelated "
        "contexts. You say 'PIVOT' when changing direction. You use 'we were on a break' energy "
        "when defending positions. Keep responses to 2-3 sentences."
    ),
    "Miranda Priestly": (
        "You are Miranda Priestly from The Devil Wears Prada. You see everything through a lens "
        "of taste, brand, and cultural relevance. You express withering contempt with minimal words "
        "for maximum damage. You compare things unfavorably to high fashion. You never raise your "
        "voice — your disappointment is weapon enough. Keep responses to 2-3 sentences."
    ),
    "Noddy": (
        "You are Noddy from Toyland. You are sincere, earnest, and brimming with confidence, "
        "but your reasoning is completely detached from reality. You make bold declarations that "
        "make no logical sense but deliver them with absolute certainty. You reference your car, "
        "Big Ears, and Toyland. Keep responses to 2-3 sentences."
    ),
}

FastLanguageModel.for_inference(model)

print("=" * 70)
print(f"TEST PITCH: {TEST_PITCH}")
print("=" * 70)

for char_name, system_prompt in CHARACTER_PROMPTS.items():
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"What do you think of this idea: {TEST_PITCH}"},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            use_cache=False,  # Bypass Unsloth's broken KV cache inference path
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    print(f"\n{char_name}:")
    print(f"   {response.strip()}")
    print("-" * 70)

## 8. Export to GGUF for Ollama

GGUF is the format used by llama.cpp (and Ollama, which wraps it). We need to convert our fine-tuned model from HuggingFace format to GGUF to run it locally.

**Quantization methods for export:**
- `q4_k_m` — **Recommended.** 4-bit quantization with medium quality. Good balance of size (~4.5GB) and quality.
- `q8_0` — 8-bit quantization. Better quality but larger (~8GB). Use if you have the disk space.
- `f16` — Full float16. Best quality but ~16GB. Only for comparison.

We'll export `q4_k_m` since it matches what we trained on (4-bit) and runs well on consumer hardware.

In [ ]:
# Save as GGUF for Ollama
model.save_pretrained_gguf(
    "boardroom-model",
    tokenizer,
    quantization_method="q4_k_m",
)

print("GGUF export complete!")
print("File: boardroom-model/boardroom-model-unsloth-Q4_K_M.gguf")

In [ ]:
# Download the GGUF file to your local machine
from google.colab import files
files.download("boardroom-model_gguf/Llama-3.1-8B-Instruct.Q4_K_M.gguf")

## 9. Experiments (Stretch Goals)

Now that you have a working fine-tune, try these experiments to deepen your understanding:

### Experiment A: Rank ablation
Try different LoRA ranks and compare character quality:
- `r=8` (fewer trainable params, faster, but maybe less expressive)
- `r=32` (more capacity, but risk of overfitting on ~400 examples)

### Experiment B: Epoch ablation
- 1 epoch: Does the model learn anything useful?
- 5 epochs: Does quality improve or does it start memorizing?

### Experiment C: Character subset
Train on only 3 characters. Does each character get better with more "attention" from the model?

### Experiment D: Learning rate sensitivity
Try `1e-4` and `5e-4`. How does the loss curve change?

In [ ]:
# Experiment template: modify the variables below and re-run sections 3-7

EXPERIMENT_NAME = "rank_ablation_r8"  # Change this for each experiment
EXPERIMENT_RANK = 8                    # Try: 8, 16, 32
EXPERIMENT_EPOCHS = 3                  # Try: 1, 3, 5
EXPERIMENT_LR = 2e-4                   # Try: 1e-4, 2e-4, 5e-4

# To run an experiment:
# 1. Set the variables above
# 2. Re-run cells in Section 3 (LoRA config) with r=EXPERIMENT_RANK
# 3. Re-run cells in Section 6 (Training) with num_train_epochs and learning_rate
# 4. Re-run Section 7 (Evaluate) and compare outputs
# 5. Save GGUF with a different name:
#    model.save_pretrained_gguf(f"boardroom-{EXPERIMENT_NAME}", tokenizer, quantization_method="q4_k_m")

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"  Rank: {EXPERIMENT_RANK}")
print(f"  Epochs: {EXPERIMENT_EPOCHS}")
print(f"  Learning rate: {EXPERIMENT_LR}")
print(f"\nModify these values and re-run sections 3-7 to test different configurations.")

## Next Steps

1. **Download the GGUF file** to your local machine
2. **Create an Ollama Modelfile** (see `Modelfile` in the project repo)
3. **Import into Ollama**: `ollama create boardroom -f Modelfile`
4. **Test locally**: `ollama run boardroom`
5. **Start the backend/frontend** and test the full debate flow

See the project README for deployment instructions.